
### Create Three-Minute Sensor Windows


In [31]:

import numpy as np
import pandas as pd
from pathlib import Path

# Locate the project directory
project_root = Path.cwd()

if not (project_root / "data/processed/cleaned_activity_sessions.csv").exists():
    project_root = project_root.parent

# Load the cleaned sensor readings
data_path = project_root / "data/processed/cleaned_activity_sessions.csv"
df = pd.read_csv(data_path)

# Define a three-minute window (180 seconds)
window_seconds = 180

# Keep participant and activity identifiers
group_columns = ["Task", "Participant", "Session"]

# Assign a window number within each activity session
df["Window_ID"] = (
    df["Elapsed_Time_s"] // window_seconds
).astype(int)

# Count sensor readings per window
window_summary = (
    df.groupby(group_columns + ["Window_ID"])
    .agg(
        Readings=("Time", "size"),
        Start_Time_s=("Elapsed_Time_s", "min"),
        End_Time_s=("Elapsed_Time_s", "max")
    )
    .reset_index()
)

# Identify complete windows at the dataset's 2 Hz sampling rate
expected_readings = 360

window_summary["Complete_Window"] = (
    window_summary["Readings"] >= expected_readings
)

print("Original activity sessions:", df[group_columns].drop_duplicates().shape[0])
print("Total windows:", len(window_summary))
print("Complete windows:", window_summary["Complete_Window"].sum())
print("Incomplete windows:", (~window_summary["Complete_Window"]).sum())

display(window_summary.head(15))


Original activity sessions: 334
Total windows: 961
Complete windows: 627
Incomplete windows: 334


,Task,Participant,Session,Window_ID,Readings,Start_Time_s,End_Time_s,Complete_Window
0,Composite,P1,Rep1,0,360,0.0,179.5,True
1,Composite,P1,Rep1,1,360,180.0,359.5,True
2,Composite,P1,Rep1,2,309,360.0,514.0,False
3,Composite,P1,Rep2,0,360,0.0,179.5,True
4,Composite,P1,Rep2,1,360,180.0,359.5,True
5,Composite,P1,Rep2,2,147,360.0,433.0,False
6,Composite,P1,Rep3,0,360,0.0,179.5,True
7,Composite,P1,Rep3,1,360,180.0,359.5,True
8,Composite,P1,Rep3,2,106,360.0,412.5,False
9,Composite,P1,Rep4,0,360,0.0,179.5,True



### Analyze Window Distribution


In [32]:

# Count complete and incomplete windows
print("Total windows:", len(window_summary))
print("Complete windows:", window_summary["Complete_Window"].sum())
print("Incomplete windows:", (~window_summary["Complete_Window"]).sum())

# Examine the number of readings in incomplete windows
incomplete = window_summary[~window_summary["Complete_Window"]]

print("\nIncomplete Window Statistics:")
display(incomplete["Readings"].describe().round(2))

# Check how many incomplete windows contain substantial data
print("\nIncomplete windows with at least 180 readings:",
      (incomplete["Readings"] >= 180).sum())

print("Incomplete windows with fewer than 180 readings:",
      (incomplete["Readings"] < 180).sum())


Total windows: 961
Complete windows: 627
Incomplete windows: 334

Incomplete Window Statistics:


count    334.00
mean     182.99
std       99.72
min        1.00
25%      101.25
50%      184.50
75%      260.50
max      358.00
Name: Readings, dtype: float64


Incomplete windows with at least 180 readings: 169
Incomplete windows with fewer than 180 readings: 165



### Select Complete Sensor Windows


In [33]:

# Identify windows containing 360 sensor readings
complete_windows = window_summary.loc[
    window_summary["Complete_Window"],
    group_columns + ["Window_ID"]
]

# Keep sensor readings belonging to complete windows
window_df = df.merge(
    complete_windows,
    on=group_columns + ["Window_ID"],
    how="inner",
    validate="many_to_one"
)

# Verify the selected windows
print("Complete windows:", len(complete_windows))
print("Selected sensor readings:", len(window_df))
print("Participants:", window_df["Participant"].nunique())

print(
    "Activity sessions represented:",
    window_df[group_columns].drop_duplicates().shape[0]
)

display(window_df.head())


Complete windows: 627
Selected sensor readings: 225720
Participants: 38
Activity sessions represented: 334


,Task,Participant,Session,Time,HR_Processed,HRV,Temperature,IMU_5_ax_g_,IMU_5_ay_g_,IMU_5_az_g_,IMU_5_gx_dps_,IMU_5_gy_dps_,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final,Elapsed_Time_s,Window_ID
0,Composite,P1,Rep1,522500,102.5,22.402814,34.224213,0.042525,0.129985,0.011618,14.436704,14.436704,14.436704,1,3,0.0,0
1,Composite,P1,Rep1,523000,104.7,22.376775,34.226817,-0.061162,-2.334652,0.173983,1.233281,1.233281,1.233281,1,3,0.5,0
2,Composite,P1,Rep1,523500,108.2,22.351200,34.228885,0.348357,-2.585750,0.150009,13.288418,13.288418,13.288418,1,3,1.0,0
3,Composite,P1,Rep1,524000,108.2,22.326121,34.230427,0.020237,-0.042484,0.012974,-34.480159,-34.480159,-34.480159,1,3,1.5,0
4,Composite,P1,Rep1,524500,108.2,22.301569,34.231472,0.039516,-0.311922,0.089224,-5.812640,-5.812640,-5.812640,1,3,2.0,0



## Estimate Window-Level Fatigue Scores


In [34]:

# Identify each participant's activity session
group_columns = ["Task", "Participant", "Session"]
window_columns = group_columns + ["Window_ID"]

# Get initial fatigue, final fatigue, and total session duration
session_info = df.groupby(
    group_columns, as_index=False
).agg(
    Initial_Fatigue=("Physical Fatigue - Initial", "first"),
    Final_Fatigue=("Physical Fatigue - Final", "first"),
    Duration_s=("Elapsed_Time_s", "max")
)

# Calculate the midpoint of each complete 3-minute window
window_times = window_df.groupby(
    window_columns, as_index=False
).agg(
    Window_Midpoint_s=("Elapsed_Time_s", "mean")
)

# Combine each window with its original fatigue scores
window_targets = window_times.merge(
    session_info,
    on=group_columns,
    how="left",
    validate="many_to_one"
)

# Estimate fatigue at the midpoint of each window
window_targets["Estimated_Fatigue"] = (
    window_targets["Initial_Fatigue"] +
    (
        window_targets["Final_Fatigue"] -
        window_targets["Initial_Fatigue"]
    ) *
    (
        window_targets["Window_Midpoint_s"] /
        window_targets["Duration_s"]
    )
).clip(0, 10)

# Verify calculated targets
print("Total window targets:", len(window_targets))
print(
    "Missing estimated targets:",
    window_targets["Estimated_Fatigue"].isna().sum()
)

display(window_targets.head(10))


Total window targets: 627
Missing estimated targets: 0


,Task,Participant,Session,Window_ID,Window_Midpoint_s,Initial_Fatigue,Final_Fatigue,Duration_s,Estimated_Fatigue
0,Composite,P1,Rep1,0,89.75,1,3,514.0,1.349222
1,Composite,P1,Rep1,1,269.75,1,3,514.0,2.049611
2,Composite,P1,Rep2,0,89.75,3,5,433.0,3.414550
3,Composite,P1,Rep2,1,269.75,3,5,433.0,4.245958
4,Composite,P1,Rep3,0,89.75,5,6,412.5,5.217576
5,Composite,P1,Rep3,1,269.75,5,6,412.5,5.653939
6,Composite,P1,Rep4,0,89.75,6,7,393.5,6.228081
7,Composite,P1,Rep4,1,269.75,6,7,393.5,6.685515
8,Composite,P1,Rep5,0,89.75,7,7,348.0,7.000000
9,Composite,P10,Rep1,0,89.75,0,1,783.0,0.114623



### Extract Window-Level Sensor Features


In [35]:

import numpy as np

# Create acceleration magnitude using three accelerometer axes
window_df["Acceleration_Magnitude"] = np.sqrt(
    window_df["IMU_5_ax_g_"] ** 2 +
    window_df["IMU_5_ay_g_"] ** 2 +
    window_df["IMU_5_az_g_"] ** 2
)

# Use one gyroscope axis as a provisional movement signal
# The dataset's three gyroscope axes contain identical values
window_df["Gyroscope_Proxy"] = window_df["IMU_5_gx_dps_"].abs()

# Define the five sensor signals
sensor_features = {
    "HR": "HR_Processed",
    "HRV": "HRV",
    "Temperature": "Temperature",
    "Acceleration": "Acceleration_Magnitude",
    "Gyroscope": "Gyroscope_Proxy"
}

# Define statistical features for each sensor
statistics = {
    "Mean": "mean",
    "Median": "median",
    "STD": "std",
    "P10": lambda x: x.quantile(0.10),
    "P90": lambda x: x.quantile(0.90)
}

# Build feature extraction rules
aggregations = {
    f"{name}_{stat}": (column, function)
    for name, column in sensor_features.items()
    for stat, function in statistics.items()
}

# Extract features for every 3-minute window
window_features = (
    window_df.groupby(window_columns, as_index=False)
    .agg(**aggregations)
)

# Attach estimated fatigue scores as the prediction target
window_features = window_features.merge(
    window_targets[window_columns + ["Estimated_Fatigue"]],
    on=window_columns,
    how="left",
    validate="one_to_one"
)

# Identify input features
feature_columns = list(aggregations.keys())

print("Total windows:", len(window_features))
print("Total input features:", len(feature_columns))
print("Participants:", window_features["Participant"].nunique())
print("Missing target values:", window_features["Estimated_Fatigue"].isna().sum())
print("Missing feature values:", window_features[feature_columns].isna().sum().sum())

display(window_features.head())


Total windows: 627
Total input features: 25
Participants: 38
Missing target values: 0
Missing feature values: 155


,Task,Participant,Session,Window_ID,HR_Mean,HR_Median,HR_STD,HR_P10,HR_P90,HRV_Mean,...,Acceleration_Median,Acceleration_STD,Acceleration_P10,Acceleration_P90,Gyroscope_Mean,Gyroscope_Median,Gyroscope_STD,Gyroscope_P10,Gyroscope_P90,Estimated_Fatigue
0,Composite,P1,Rep1,0,126.711688,128.80,6.693684,108.20,131.30,19.778580,...,0.982140,0.156706,0.913268,1.038046,8.042003,5.368445,7.962698,0.907790,19.725297,1.349222
1,Composite,P1,Rep1,1,133.483333,134.00,3.611966,128.09,137.60,33.069785,...,0.987280,0.091194,0.919348,1.031999,7.384589,4.866925,7.270962,0.632540,18.645361,2.049611
2,Composite,P1,Rep2,0,133.513056,134.65,4.603060,126.78,138.50,14.556046,...,0.980623,0.124468,0.912352,1.039527,8.439136,6.163824,8.134527,0.773789,20.462939,3.414550
3,Composite,P1,Rep2,1,140.258333,140.10,2.853847,136.77,144.12,12.045454,...,0.980429,0.100610,0.925767,1.031780,7.226054,4.920109,6.934532,0.953933,16.512819,4.245958
4,Composite,P1,Rep3,0,134.691944,134.45,6.299827,126.80,143.61,21.017761,...,0.981958,0.133891,0.927608,1.031373,8.584598,5.537237,8.474195,0.879740,20.929297,5.217576



### Split Window-Level Training and Testing Data


In [36]:

from sklearn.model_selection import GroupShuffleSplit

# Define input features and regression target
X = window_features[feature_columns].copy()
y = window_features["Estimated_Fatigue"].copy()

# Group by participant to prevent data leakage
groups = window_features["Participant"]

# Split participants into training and testing groups
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Create training and testing datasets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Check participant separation
train_participants = set(groups.iloc[train_idx])
test_participants = set(groups.iloc[test_idx])

print("Training windows:", len(X_train))
print("Testing windows:", len(X_test))
print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))
print(
    "Shared participants:",
    len(train_participants & test_participants)
)

print("\nTraining target range:", y_train.min(), "-", y_train.max())
print("Testing target range:", y_test.min(), "-", y_test.max())

# Verify that no participant appears in both datasets
assert train_participants.isdisjoint(test_participants)


Training windows: 500
Testing windows: 127
Training participants: 30
Testing participants: 8
Shared participants: 0

Training target range: 0.0 - 10.0
Testing target range: 0.0 - 8.563740856844305



### Train Window-Based XGBoost Regressor


In [37]:

from xgboost import XGBRegressor

# Initialize XGBoost for fatigue score prediction
xgb_window_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_lambda=5,
    tree_method="hist",
    random_state=42,
    n_jobs=4
)

# Train using 3-minute window features
xgb_window_model.fit(X_train, y_train)

print("XGBoost Regressor training completed!")
print("Training windows:", len(X_train))
print("Input features:", X_train.shape[1])
print("Target: Estimated physical fatigue score (0–10)")
print("Trees:", xgb_window_model.n_estimators)


XGBoost Regressor training completed!
Training windows: 500
Input features: 25
Target: Estimated physical fatigue score (0–10)
Trees: 300



### Evaluate Window-Based XGBoost Regressor


In [38]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# Predict fatigue scores for unseen participants
y_pred = xgb_window_model.predict(X_test)

# Restrict predictions to the valid fatigue scale (0–10)
y_pred = np.clip(y_pred, 0, 10)

# Evaluate regression performance
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("REGRESSION PERFORMANCE")
print("Test windows:", len(y_test))
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

# Convert continuous fatigue scores into fatigue categories
# Low: <4, Moderate: 4 to <7, High: >=7
def fatigue_category(scores):
    scores = np.asarray(scores)
    return np.where(
        scores < 4, "Low",
        np.where(scores < 7, "Moderate", "High")
    )

y_true_classes = fatigue_category(y_test)
y_pred_classes = fatigue_category(y_pred)

print("\nCLASSIFICATION PERFORMANCE")
print(f"Accuracy: {accuracy_score(y_true_classes, y_pred_classes):.2%}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true_classes, y_pred_classes):.2%}")
print(f"Macro F1: {f1_score(y_true_classes, y_pred_classes, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true_classes,
    y_pred_classes,
    labels=["Low", "Moderate", "High"],
    zero_division=0
))

# Show sample predictions
results_df = pd.DataFrame({
    "Actual_Fatigue": y_test.to_numpy(),
    "Predicted_Fatigue": y_pred,
    "Actual_Level": y_true_classes,
    "Predicted_Level": y_pred_classes
})

display(results_df.head(10).round(2))


REGRESSION PERFORMANCE
Test windows: 127
MAE: 1.9183
RMSE: 2.2287
R² Score: 0.1606

CLASSIFICATION PERFORMANCE
Accuracy: 58.27%
Balanced Accuracy: 47.47%
Macro F1: 0.4202

Classification Report:
              precision    recall  f1-score   support

         Low       0.78      0.59      0.67        66
    Moderate       0.45      0.83      0.59        42
        High       0.00      0.00      0.00        19

    accuracy                           0.58       127
   macro avg       0.41      0.47      0.42       127
weighted avg       0.56      0.58      0.54       127



,Actual_Fatigue,Predicted_Fatigue,Actual_Level,Predicted_Level
0,5.00,4.05,Moderate,Moderate
1,5.00,4.71,Moderate,Moderate
2,5.27,2.65,Moderate,Low
3,6.00,5.23,Moderate,Moderate
4,6.36,2.86,Moderate,Low
5,7.00,5.08,High,Moderate
6,1.31,3.85,Low,Low
7,1.93,3.68,Low,Low
8,2.54,3.18,Low,Low
9,3.00,4.82,Low,Moderate



### Analyze High-Fatigue Prediction Failure


In [39]:

# Examine the range of predicted fatigue scores
print("Minimum predicted fatigue:", results_df["Predicted_Fatigue"].min())
print("Maximum predicted fatigue:", results_df["Predicted_Fatigue"].max())

# Compare actual and predicted fatigue categories
print("\nActual fatigue distribution:")
print(results_df["Actual_Level"].value_counts())

print("\nPredicted fatigue distribution:")
print(results_df["Predicted_Level"].value_counts())

# Display confusion matrix
class_names = ["Low", "Moderate", "High"]

confusion = pd.crosstab(
    results_df["Actual_Level"],
    results_df["Predicted_Level"],
    rownames=["Actual"],
    colnames=["Predicted"]
).reindex(
    index=class_names,
    columns=class_names,
    fill_value=0
)

print("\nConfusion Matrix:")
display(confusion)

# Examine predictions specifically for high-fatigue windows
print("\nHigh-Fatigue Predictions:")

display(
    results_df.loc[
        results_df["Actual_Level"] == "High",
        ["Actual_Fatigue", "Predicted_Fatigue"]
    ].sort_values("Actual_Fatigue", ascending=False)
)


Minimum predicted fatigue: 2.136647
Maximum predicted fatigue: 6.9560266

Actual fatigue distribution:
Actual_Level
Low         66
Moderate    42
High        19
Name: count, dtype: int64

Predicted fatigue distribution:
Predicted_Level
Moderate    77
Low         50
Name: count, dtype: int64

Confusion Matrix:


Predicted,Low,Moderate,High
Actual,,,
Low,39,27,0
Moderate,7,35,0
High,4,15,0



High-Fatigue Predictions:


,Actual_Fatigue,Predicted_Fatigue
126,8.563741,3.942666
50,8.294262,4.122837
125,8.187565,4.432641
124,8.000000,5.060106
122,8.000000,4.081785
49,8.000000,3.245398
20,8.000000,5.886752
123,8.000000,2.804404
19,7.746196,3.934010
104,7.741550,5.676697



### Analyze Training Fatigue Distribution


In [40]:

# Predict fatigue scores on training data
train_predictions = np.clip(
    xgb_window_model.predict(X_train),
    0,
    10
)

# Convert training scores into fatigue categories
train_actual_levels = fatigue_category(y_train)
train_predicted_levels = fatigue_category(train_predictions)

# Compare actual training labels with model predictions
print("ACTUAL TRAINING DISTRIBUTION")
print(pd.Series(train_actual_levels).value_counts())

print("\nPREDICTED TRAINING DISTRIBUTION")
print(pd.Series(train_predicted_levels).value_counts())

# Check whether the model can predict high fatigue in training
print("\nTRAINING PREDICTION RANGE")
print("Minimum:", train_predictions.min())
print("Maximum:", train_predictions.max())

# Compare regression performance between training and testing
print("\nTRAINING VS TESTING MAE")
print(
    "Training MAE:",
    round(mean_absolute_error(y_train, train_predictions), 4)
)
print(
    "Testing MAE:",
    round(mean_absolute_error(y_test, y_pred), 4)
)


ACTUAL TRAINING DISTRIBUTION
Low         220
Moderate    194
High         86
Name: count, dtype: int64

PREDICTED TRAINING DISTRIBUTION
Moderate    232
Low         229
High         39
Name: count, dtype: int64

TRAINING PREDICTION RANGE
Minimum: 0.6441863
Maximum: 8.861985

TRAINING VS TESTING MAE
Training MAE: 0.6465
Testing MAE: 1.9183



### Evaluate Participant-Based Cross-Validation


In [41]:

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

# Use only training data for cross-validation
X_cv = X_train.reset_index(drop=True)
y_cv = y_train.reset_index(drop=True)
participant_cv = groups.iloc[train_idx].reset_index(drop=True)

# Create five participant-independent folds
group_kfold = GroupKFold(n_splits=5)

cv_results = []

for fold, (train_ids, val_ids) in enumerate(
    group_kfold.split(X_cv, y_cv, groups=participant_cv),
    start=1
):
    X_fold_train = X_cv.iloc[train_ids]
    X_fold_val = X_cv.iloc[val_ids]

    y_fold_train = y_cv.iloc[train_ids]
    y_fold_val = y_cv.iloc[val_ids]

    # Train a separate model for this fold
    fold_model = clone(xgb_window_model)
    fold_model.fit(X_fold_train, y_fold_train)

    # Predict fatigue scores for unseen participants
    predictions = np.clip(
        fold_model.predict(X_fold_val), 0, 10
    )

    # Compare against a simple baseline
    baseline = np.full(
        len(y_fold_val),
        y_fold_train.median()
    )

    # Measure high-fatigue detection
    actual_high = y_fold_val.to_numpy() >= 7
    predicted_high = predictions >= 7

    high_recall = (
        predicted_high[actual_high].mean()
        if actual_high.sum() > 0
        else np.nan
    )

    cv_results.append({
        "Fold": fold,
        "Validation_MAE": mean_absolute_error(
            y_fold_val, predictions
        ),
        "Baseline_MAE": mean_absolute_error(
            y_fold_val, baseline
        ),
        "High_Recall": high_recall
    })

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))

print("Average Validation MAE:",
      round(cv_results_df["Validation_MAE"].mean(), 4))

print("Average Baseline MAE:",
      round(cv_results_df["Baseline_MAE"].mean(), 4))

print("Average High-Fatigue Recall:",
      round(cv_results_df["High_Recall"].mean(), 4))


,Fold,Validation_MAE,Baseline_MAE,High_Recall
0,1,1.9575,1.6433,0.0000
1,2,1.9450,1.8090,0.0625
2,3,2.5650,2.4927,0.0000
3,4,1.8898,1.7847,0.0000
4,5,1.9918,1.9868,0.0000


Average Validation MAE: 2.0698
Average Baseline MAE: 1.9433
Average High-Fatigue Recall: 0.0125



### Investigate Sensor Data Quality


In [42]:

import numpy as np
import pandas as pd

# Check missing window-level features
missing_features = (
    window_features[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("MISSING FEATURES")
display(missing_features[missing_features > 0])

# Examine the distribution of raw HR and HRV readings
print("\nHR AND HRV DISTRIBUTION")

display(
    window_df[["HR_Processed", "HRV"]]
    .describe(percentiles=[0.01, 0.50, 0.95, 0.99, 0.999])
    .round(2)
)

# Check missing sensor readings
print("\nMISSING SENSOR READINGS")
print("Missing HR:", window_df["HR_Processed"].isna().sum())
print("Missing HRV:", window_df["HRV"].isna().sum())

# Flag unusually large HRV values for investigation
# 1000 is a diagnostic threshold, not a validated cutoff
print("\nHRV VALUES ABOVE 1000:")
print((window_df["HRV"] > 1000).sum())

# Check whether gyroscope axes are duplicates
gyro_columns = [
    "IMU_5_gx_dps_",
    "IMU_5_gy_dps_",
    "IMU_5_gz_dps_"
]

identical_gyro = (
    window_df[gyro_columns].nunique(axis=1) == 1
)

print("\nGYROSCOPE DATA QUALITY")
print("Rows with identical X, Y, Z:",
      identical_gyro.sum())

print("Total rows:", len(window_df))

# Check which features the trained model relies on
print("\nTOP 10 MODEL FEATURES")

feature_importance = pd.Series(
    xgb_window_model.feature_importances_,
    index=feature_columns
).sort_values(ascending=False)

display(feature_importance.head(10))


MISSING FEATURES


HR_Mean      31
HR_Median    31
HR_STD       31
HR_P10       31
HR_P90       31
dtype: int64


HR AND HRV DISTRIBUTION


,HR_Processed,HRV
count,185319.00,2.216220e+05
mean,104.67,8.985716e+04
std,16.08,2.804102e+06
min,40.40,0.000000e+00
1%,72.70,6.120000e+00
50%,103.90,6.121000e+01
95%,135.70,1.662370e+03
99%,148.40,4.666302e+04
99.9%,160.00,2.786095e+07
max,232.40,1.503864e+08



MISSING SENSOR READINGS
Missing HR: 40401
Missing HRV: 4098

HRV VALUES ABOVE 1000:
15175

GYROSCOPE DATA QUALITY
Rows with identical X, Y, Z: 225720
Total rows: 225720

TOP 10 MODEL FEATURES


HR_Median              0.066247
HR_Mean                0.060104
Acceleration_Median    0.054458
HR_P90                 0.047908
Temperature_P90        0.046914
HR_STD                 0.044770
Gyroscope_Mean         0.043548
Gyroscope_P10          0.043054
Gyroscope_P90          0.042729
Acceleration_STD       0.042576
dtype: float32


### Clean Extreme HRV Values


In [43]:

# Create a separate copy to preserve original sensor data
cleaned_window_df = window_df.copy()

# Experimentally flag extreme HRV readings
# Threshold is provisional, not a validated physiological cutoff
hrv_limit = 1000

extreme_hrv = cleaned_window_df["HRV"] > hrv_limit

print("Extreme HRV readings:", extreme_hrv.sum())

# Replace flagged HRV readings with missing values
cleaned_window_df.loc[extreme_hrv, "HRV"] = np.nan

# Recalculate HRV statistics for each 3-minute window
hrv_aggregations = {
    "HRV_Mean": ("HRV", "mean"),
    "HRV_Median": ("HRV", "median"),
    "HRV_STD": ("HRV", "std"),
    "HRV_P10": ("HRV", lambda x: x.quantile(0.10)),
    "HRV_P90": ("HRV", lambda x: x.quantile(0.90))
}

clean_hrv_features = (
    cleaned_window_df.groupby(
        window_columns,
        as_index=False
    )
    .agg(**hrv_aggregations)
)

# Replace only HRV features in the original window dataset
clean_window_features = (
    window_features
    .drop(columns=list(hrv_aggregations.keys()))
    .merge(
        clean_hrv_features,
        on=window_columns,
        how="left",
        validate="one_to_one"
    )
)

# Preserve the original 25 input features and their order
clean_window_features = clean_window_features[
    window_columns + feature_columns + ["Estimated_Fatigue"]
]

print("Total windows:", len(clean_window_features))
print("Input features:", len(feature_columns))
print(
    "Missing feature values:",
    clean_window_features[feature_columns].isna().sum().sum()
)

print(
    "Windows without usable HRV:",
    clean_window_features["HRV_Mean"].isna().sum()
)

display(clean_window_features.head())


Extreme HRV readings: 15175
Total windows: 627
Input features: 25
Missing feature values: 242
Windows without usable HRV: 17


,Task,Participant,Session,Window_ID,HR_Mean,HR_Median,HR_STD,HR_P10,HR_P90,HRV_Mean,...,Acceleration_Median,Acceleration_STD,Acceleration_P10,Acceleration_P90,Gyroscope_Mean,Gyroscope_Median,Gyroscope_STD,Gyroscope_P10,Gyroscope_P90,Estimated_Fatigue
0,Composite,P1,Rep1,0,126.711688,128.80,6.693684,108.20,131.30,19.778580,...,0.982140,0.156706,0.913268,1.038046,8.042003,5.368445,7.962698,0.907790,19.725297,1.349222
1,Composite,P1,Rep1,1,133.483333,134.00,3.611966,128.09,137.60,33.069785,...,0.987280,0.091194,0.919348,1.031999,7.384589,4.866925,7.270962,0.632540,18.645361,2.049611
2,Composite,P1,Rep2,0,133.513056,134.65,4.603060,126.78,138.50,14.556046,...,0.980623,0.124468,0.912352,1.039527,8.439136,6.163824,8.134527,0.773789,20.462939,3.414550
3,Composite,P1,Rep2,1,140.258333,140.10,2.853847,136.77,144.12,12.045454,...,0.980429,0.100610,0.925767,1.031780,7.226054,4.920109,6.934532,0.953933,16.512819,4.245958
4,Composite,P1,Rep3,0,134.691944,134.45,6.299827,126.80,143.61,21.017761,...,0.981958,0.133891,0.927608,1.031373,8.584598,5.537237,8.474195,0.879740,20.929297,5.217576



### Compare Original and Cleaned HRV Models


In [44]:

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

# Verify that original and cleaned windows have identical ordering
assert window_features[window_columns].reset_index(drop=True).equals(
    clean_window_features[window_columns].reset_index(drop=True)
)

# Prepare original and cleaned training features
X_original_cv = (
    window_features.iloc[train_idx][feature_columns]
    .reset_index(drop=True)
)

X_clean_cv = (
    clean_window_features.iloc[train_idx][feature_columns]
    .reset_index(drop=True)
)

y_cv = y_train.reset_index(drop=True)
participant_cv = groups.iloc[train_idx].reset_index(drop=True)

# Use the same participant-based folds for both models
group_kfold = GroupKFold(n_splits=5)

comparison_results = []

for fold, (train_ids, val_ids) in enumerate(
    group_kfold.split(X_original_cv, y_cv, participant_cv),
    start=1
):
    y_fold_train = y_cv.iloc[train_ids]
    y_fold_val = y_cv.iloc[val_ids]

    fold_result = {"Fold": fold}

    for version, X_data in [
        ("Original", X_original_cv),
        ("Cleaned", X_clean_cv)
    ]:
        # Train a fresh model
        model = clone(xgb_window_model)

        model.fit(
            X_data.iloc[train_ids],
            y_fold_train
        )

        # Predict scores for unseen participants
        predictions = np.clip(
            model.predict(X_data.iloc[val_ids]),
            0,
            10
        )

        # Calculate prediction error
        fold_result[f"{version}_MAE"] = mean_absolute_error(
            y_fold_val,
            predictions
        )

        # Calculate high-fatigue recall
        actual_high = y_fold_val.to_numpy() >= 7
        predicted_high = predictions >= 7

        fold_result[f"{version}_High_Recall"] = (
            predicted_high[actual_high].mean()
            if actual_high.sum() > 0
            else np.nan
        )

    comparison_results.append(fold_result)

comparison_df = pd.DataFrame(comparison_results)

print("CROSS-VALIDATION COMPARISON")
display(comparison_df.round(4))

print("\nAVERAGE RESULTS")
print("Original MAE:", round(comparison_df["Original_MAE"].mean(), 4))
print("Cleaned MAE:", round(comparison_df["Cleaned_MAE"].mean(), 4))

print(
    "Original High Recall:",
    round(comparison_df["Original_High_Recall"].mean(), 4)
)
print(
    "Cleaned High Recall:",
    round(comparison_df["Cleaned_High_Recall"].mean(), 4)
)


CROSS-VALIDATION COMPARISON


,Fold,Original_MAE,Original_High_Recall,Cleaned_MAE,Cleaned_High_Recall
0,1,1.9575,0.0000,1.9665,0.0000
1,2,1.9450,0.0625,1.9587,0.0625
2,3,2.5650,0.0000,2.5445,0.0000
3,4,1.8898,0.0000,1.9235,0.0000
4,5,1.9918,0.0000,1.9713,0.0000



AVERAGE RESULTS
Original MAE: 2.0698
Cleaned MAE: 2.0729
Original High Recall: 0.0125
Cleaned High Recall: 0.0125



### Save Trained XGBoost Model


In [45]:

import json

# Create directory for deployment models
model_dir = project_root / "deployment" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# Define output files
model_path = model_dir / "fatigue_xgboost.json"
metadata_path = model_dir / "fatigue_model_metadata.json"

# Save the trained XGBoost regressor
xgb_window_model.save_model(str(model_path))

# Save model configuration for the prototype
metadata = {
    "model_type": "XGBRegressor",
    "target": "Estimated_Fatigue",
    "window_seconds": 180,
    "expected_readings": 360,
    "sampling_interval_seconds": 0.5,
    "feature_columns": feature_columns,
    "fatigue_thresholds": {
        "Low": [0, 4],
        "Moderate": [4, 7],
        "High": [7, 10]
    },
    "hrv_cleaning": "Original features without experimental HRV filtering",
    "gyroscope": "Absolute value of gyroscope X axis"
}

# Save model metadata
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

# Verify that the saved model can be loaded
loaded_model = XGBRegressor()
loaded_model.load_model(str(model_path))

# Verify predictions match the original model
original_predictions = xgb_window_model.predict(X_test.head(5))
loaded_predictions = loaded_model.predict(X_test.head(5))

np.testing.assert_allclose(
    original_predictions,
    loaded_predictions,
    rtol=1e-6
)

print("Model saved successfully!")
print("Model:", model_path)
print("Metadata:", metadata_path)
print("Features:", len(feature_columns))
print("Model reload verification: Passed")


Model saved successfully!
Model: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\deployment\models\fatigue_xgboost.json
Metadata: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\deployment\models\fatigue_model_metadata.json
Features: 25
Model reload verification: Passed



### Test Prediction Backend


In [46]:

import sys
import importlib

# Add project root to Python path
sys.path.insert(0, str(project_root))

# Import and reload the latest version of predict.py
backend = importlib.import_module("deployment.predict")
backend = importlib.reload(backend)

print("Backend file:", backend.__file__)
print("Function available:", hasattr(backend, "predict_fatigue"))

# Select one complete 3-minute window
sample = window_df.iloc[:360].copy()

# Verify all readings belong to the same window
assert sample[window_columns].drop_duplicates().shape[0] == 1

# Run prediction
result = backend.predict_fatigue(sample)

print("Fatigue Score:", result["fatigue_score"])
print("Fatigue Level:", result["fatigue_level"])
print("Extracted Features:", result["features"].shape[1])


Backend file: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\deployment\predict.py
Function available: True
Fatigue Score: 3.12
Fatigue Level: Low
Extracted Features: 25



# Prepare Raw Sensor Readings for Model Comparison



### Define Raw Sensor Inputs


In [47]:

# Select the five sensor signals
raw_sensor_columns = [
    "HR_Processed",
    "HRV",
    "Temperature",
    "Acceleration_Magnitude",
    "Gyroscope_Proxy"
]

# Verify the sensor columns exist
assert all(
    column in window_df.columns
    for column in raw_sensor_columns
)

print("Number of sensor signals:", len(raw_sensor_columns))
print("Sensor signals:", raw_sensor_columns)


Number of sensor signals: 5
Sensor signals: ['HR_Processed', 'HRV', 'Temperature', 'Acceleration_Magnitude', 'Gyroscope_Proxy']



## Prepare Raw Sensor Sequences


In [48]:

# Group sensor readings by their 3-minute windows
window_groups = {
    key: group.sort_values("Elapsed_Time_s")
    for key, group in window_df.groupby(window_columns)
}

# Prepare a sequence for each window
raw_sequences = []

for key in window_features[window_columns].itertuples(
    index=False, name=None
):
    window = window_groups[key]

    # Verify each window contains exactly 360 readings
    assert len(window) == 360

    # Preserve all readings in chronological order
    raw_sequences.append(
        window[raw_sensor_columns].to_numpy(
            dtype=np.float32
        )
    )

# Combine all windows into a single array
X_raw_sequences = np.stack(raw_sequences)

print("Total windows:", X_raw_sequences.shape[0])
print("Readings per window:", X_raw_sequences.shape[1])
print("Sensor signals:", X_raw_sequences.shape[2])
print("Final shape:", X_raw_sequences.shape)


Total windows: 627
Readings per window: 360
Sensor signals: 5
Final shape: (627, 360, 5)



### Identify Missing Sensor Values


In [49]:

# Count missing values for each sensor
missing_counts = np.isnan(X_raw_sequences).sum(axis=(0, 1))

for sensor, count in zip(raw_sensor_columns, missing_counts):
    print(f"{sensor}: {count} missing values")

# Calculate total missing sensor values
print("\nTotal missing values:", missing_counts.sum())

# Count windows containing at least one missing value
windows_with_missing = np.isnan(X_raw_sequences).any(axis=(1, 2))

print("Windows with missing values:", windows_with_missing.sum())
print("Windows without missing values:", (~windows_with_missing).sum())


HR_Processed: 40401 missing values
HRV: 4098 missing values
Temperature: 0 missing values
Acceleration_Magnitude: 0 missing values
Gyroscope_Proxy: 0 missing values

Total missing values: 44499
Windows with missing values: 297
Windows without missing values: 330



### Split Raw Sensor Sequences


In [50]:

# Use the same participant-independent split as our XGBoost model
X_train_sequences = X_raw_sequences[train_idx].copy()
X_test_sequences = X_raw_sequences[test_idx].copy()

# Prepare fatigue score targets
y_raw = window_features["Estimated_Fatigue"].to_numpy()

y_train_raw = y_raw[train_idx]
y_test_raw = y_raw[test_idx]

# Verify participant separation
train_participants = set(
    window_features.iloc[train_idx]["Participant"]
)

test_participants = set(
    window_features.iloc[test_idx]["Participant"]
)

assert train_participants.isdisjoint(test_participants)

print("Training shape:", X_train_sequences.shape)
print("Testing shape:", X_test_sequences.shape)
print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))
print("Shared participants:", len(train_participants & test_participants))


Training shape: (500, 360, 5)
Testing shape: (127, 360, 5)
Training participants: 30
Testing participants: 8
Shared participants: 0



### Handle Missing Sensor Values


In [51]:

# Calculate one median per sensor using training data only
sensor_medians = np.nanmedian(
    X_train_sequences,
    axis=(0, 1)
)

# Replace missing values in training data
X_train_filled = np.where(
    np.isnan(X_train_sequences),
    sensor_medians,
    X_train_sequences
)

# Use the same training medians for testing data
X_test_filled = np.where(
    np.isnan(X_test_sequences),
    sensor_medians,
    X_test_sequences
)

# Verify missing values were handled
print("Training missing values:", np.isnan(X_train_filled).sum())
print("Testing missing values:", np.isnan(X_test_filled).sum())

print("\nTraining shape:", X_train_filled.shape)
print("Testing shape:", X_test_filled.shape)

print("\nSensor replacement values:")
for sensor, median in zip(raw_sensor_columns, sensor_medians):
    print(f"{sensor}: {median:.2f}")


Training missing values: 0
Testing missing values: 0

Training shape: (500, 360, 5)
Testing shape: (127, 360, 5)

Sensor replacement values:
HR_Processed: 104.30
HRV: 59.40
Temperature: 34.53
Acceleration_Magnitude: 1.03
Gyroscope_Proxy: 4.44



### Prepare Decision Tree Inputs


In [52]:

# Flatten each window into 1800 input values
X_train_dt = X_train_filled.reshape(
    X_train_filled.shape[0], -1
)

X_test_dt = X_test_filled.reshape(
    X_test_filled.shape[0], -1
)

# Verify the shapes
print("Training shape:", X_train_dt.shape)
print("Testing shape:", X_test_dt.shape)
print("Target training shape:", y_train_raw.shape)
print("Target testing shape:", y_test_raw.shape)


Training shape: (500, 1800)
Testing shape: (127, 1800)
Target training shape: (500,)
Target testing shape: (127,)



### Train Decision Tree Regressor


In [53]:

from sklearn.tree import DecisionTreeRegressor

# Initialize Decision Tree
dt_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

# Train using all 1800 input values per window
dt_model.fit(X_train_dt, y_train_raw)

print("Decision Tree training completed!")
print("Tree depth:", dt_model.get_depth())
print("Number of leaves:", dt_model.get_n_leaves())


Decision Tree training completed!
Tree depth: 5
Number of leaves: 26



### Evaluate Decision Tree Regression


In [54]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict fatigue scores
y_pred_dt = dt_model.predict(X_test_dt)

# Keep predictions within the 0–10 fatigue scale
y_pred_dt = np.clip(y_pred_dt, 0, 10)

# Calculate evaluation metrics
dt_mae = mean_absolute_error(y_test_raw, y_pred_dt)
dt_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_dt))
dt_r2 = r2_score(y_test_raw, y_pred_dt)

print(f"MAE: {dt_mae:.4f}")
print(f"RMSE: {dt_rmse:.4f}")
print(f"R² Score: {dt_r2:.4f}")


MAE: 2.2097
RMSE: 2.7051
R² Score: -0.2366



### Evaluate Decision Tree Classification Accuracy


In [55]:

from sklearn.metrics import accuracy_score

# Convert actual and predicted scores into fatigue levels
y_test_levels = np.digitize(y_test_raw, bins=[4, 7])
y_pred_dt_levels = np.digitize(y_pred_dt, bins=[4, 7])

# Calculate classification accuracy
dt_accuracy = accuracy_score(y_test_levels, y_pred_dt_levels)

print(f"Decision Tree Accuracy: {dt_accuracy * 100:.2f}%")


Decision Tree Accuracy: 48.03%



### Evaluate High Fatigue Detection


In [56]:

# Identify actual high-fatigue windows
high_mask = y_test_levels == 2

# Count correctly detected high-fatigue windows
total_high = high_mask.sum()
detected_high = (y_pred_dt_levels[high_mask] == 2).sum()

# Calculate high-fatigue recall
high_recall = detected_high / total_high if total_high > 0 else 0

print("Actual High Fatigue Windows:", total_high)
print("Correctly Detected:", detected_high)
print(f"High Fatigue Recall: {high_recall * 100:.2f}%")


Actual High Fatigue Windows: 19
Correctly Detected: 1
High Fatigue Recall: 5.26%



### Save Decision Tree 3-Minute Window Flattened Model


In [57]:

import joblib
from pathlib import Path

# Locate project folder
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create model directory
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# Save model and required input information
model_path = model_dir / "decision_tree_flattened.joblib"

joblib.dump({
    "model": dt_model,
    "sensor_columns": raw_sensor_columns,
    "sensor_medians": sensor_medians,
    "window_size": 360,
    "thresholds": [4, 7]
}, model_path)

print("Decision Tree model saved successfully!")
print("Location:", model_path)


Decision Tree model saved successfully!
Location: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\models\decision_tree_flattened.joblib



## Decision Tree — 3-Minute Window Flattened Model

### Model Configuration

- **Window Duration:** 3 minutes (180 seconds)
- **Readings per Window:** 360
- **Sensor Signals:** 5
- **Input Features:** 1,800 (Flattened)
- **Training Windows:** 500
- **Testing Windows:** 127
- **Tree Depth:** 5
- **Leaf Nodes:** 26

### Model Evaluation Results

- **Classification Accuracy:** 48.03%
- **MAE (Mean Absolute Error):** 2.2097
- **MSE (Mean Squared Error):** ≈ 7.3176
- **RMSE (Root Mean Squared Error):** 2.7051
- **R² Score:** -0.2366
- **Actual High Fatigue Windows:** 19
- **Correctly Detected High Fatigue Windows:** 1
- **High Fatigue Recall:** 5.26%



### Save Flattened Dataset for Model Comparison


In [60]:

# Save the same data used to train Decision Tree
dataset_path = project_dir / "results" / "raw_window_flattened_split.npz"

dataset_path.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    dataset_path,
    X_train=X_train_dt,
    X_test=X_test_dt,
    y_train=y_train_raw,
    y_test=y_test_raw
)

print("Dataset saved successfully!")
print("Location:", dataset_path)
print("Training:", X_train_dt.shape)
print("Testing:", X_test_dt.shape)


Dataset saved successfully!
Location: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\results\raw_window_flattened_split.npz
Training: (500, 1800)
Testing: (127, 1800)
